# EDA FakeNewsNet (Politifact + GossipCop)

Ce notebook réalise une **analyse exploratoire des données (EDA)** en fusionnant 4 fichiers :
- politifact_real.csv
- politifact_fake.csv
- gossipcop_real.csv
- gossipcop_fake.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from urllib.parse import urlparse
import re
from collections import Counter

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 140)

In [ ]:
# Chemin racine du projet (adapter si besoin)
ROOT = Path.cwd()

files = {
    "politifact_real": ROOT / "valentine/FakeNewsNet-master/dataset/politifact_real.csv",
    "politifact_fake": ROOT / "valentine/FakeNewsNet-master/dataset/politifact_fake.csv",
    "gossipcop_real": ROOT / "valentine/FakeNewsNet-master/dataset/gossipcop_real.csv",
    "gossipcop_fake": ROOT / "valentine/FakeNewsNet-master/dataset/gossipcop_fake.csv",
}

def load_with_meta(path: Path, key: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    source = key.split("_")[0]      # politifact / gossipcop
    label_str = key.split("_")[1]   # real / fake
    df["source"] = source
    df["label_str"] = label_str
    df["label"] = (label_str == "fake").astype(int)  # 1=fake, 0=real
    return df

dfs = [load_with_meta(path, key) for key, path in files.items()]
df = pd.concat(dfs, ignore_index=True)

print(f"Nombre total de lignes : {len(df):,}")
print(f"Nombre de colonnes : {df.shape[1]}")
df.head()

In [ ]:
# Vue globale
display(df.sample(5, random_state=42))
display(df.describe(include="all").T)

print("\nTypes de colonnes:")
display(df.dtypes.to_frame("dtype"))

In [ ]:
# Valeurs manquantes et doublons
missing = df.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
display(missing)

dup_all = df.duplicated().sum()
dup_id = df.duplicated(subset=["id"]).sum() if "id" in df.columns else np.nan
dup_title = df.duplicated(subset=["title"]).sum() if "title" in df.columns else np.nan
print(f"Doublons (lignes complètes) : {dup_all}")
print(f"Doublons sur id : {dup_id}")
print(f"Doublons sur title : {dup_title}")

In [ ]:
# Répartition des classes et des sources
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.countplot(data=df, x="label_str", order=["real", "fake"], ax=axes[0], palette="Set2")
axes[0].set_title("Répartition real vs fake")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Nombre")

sns.countplot(data=df, x="source", ax=axes[1], palette="Set2")
axes[1].set_title("Répartition par source")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Nombre")

ct = pd.crosstab(df["source"], df["label_str"]).reindex(columns=["real", "fake"], fill_value=0)
ct.plot(kind="bar", stacked=True, ax=axes[2], colormap="Set3")
axes[2].set_title("Classes par source (stacked)")
axes[2].set_xlabel("Source")
axes[2].set_ylabel("Nombre")
axes[2].legend(title="Classe")

plt.tight_layout()
plt.show()

display(ct)

In [ ]:
# Features textuelles simples
if "title" in df.columns:
    df["title_len_chars"] = df["title"].fillna("").str.len()
    df["title_len_words"] = df["title"].fillna("").str.split().str.len()

if "tweet_ids" in df.columns:
    # nombre de tweet IDs approximatif (séparation par espaces)
    df["n_tweet_ids"] = (
        df["tweet_ids"].fillna("").astype(str).str.strip().str.split(r"\s+").str.len()
    )

num_cols = [c for c in ["title_len_chars", "title_len_words", "n_tweet_ids"] if c in df.columns]
display(df[num_cols].describe().T)

if num_cols:
    fig, axes = plt.subplots(1, len(num_cols), figsize=(6 * len(num_cols), 4))
    if len(num_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, num_cols):
        sns.boxplot(data=df, x="label_str", y=col, order=["real", "fake"], ax=ax, palette="pastel")
        ax.set_title(f"{col} par classe")

    plt.tight_layout()
    plt.show()

In [ ]:
# Analyse des domaines d'URL
def extract_domain(url: str) -> str:
    if pd.isna(url):
        return ""
    url = str(url).strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    try:
        netloc = urlparse(url).netloc.lower()
        return netloc.replace("www.", "")
    except Exception:
        return ""

df["domain"] = df["news_url"].apply(extract_domain)

top_domains = df["domain"].value_counts().head(20)
display(top_domains.to_frame("count"))

plt.figure(figsize=(10, 6))
sns.barplot(x=top_domains.values, y=top_domains.index, palette="viridis")
plt.title("Top 20 domaines")
plt.xlabel("Nombre d'articles")
plt.ylabel("Domaine")
plt.tight_layout()
plt.show()

In [ ]:
# Mots fréquents dans les titres
stopwords_fr_en = {
    "the", "a", "an", "of", "to", "and", "in", "for", "on", "with", "at",
    "de", "la", "le", "les", "des", "du", "et", "en", "un", "une"
}

def top_words(series: pd.Series, n=20):
    text = " ".join(series.fillna("").astype(str).str.lower())
    tokens = re.findall(r"[a-zà-ÿ']{3,}", text)
    tokens = [t for t in tokens if t not in stopwords_fr_en]
    return Counter(tokens).most_common(n)

for cls in ["real", "fake"]:
    words = top_words(df.loc[df["label_str"] == cls, "title"], n=20)
    words_df = pd.DataFrame(words, columns=["word", "count"] )
    display(words_df.head(20))

    plt.figure(figsize=(9, 5))
    sns.barplot(data=words_df, x="count", y="word", palette="magma")
    plt.title(f"Top mots (titres) - {cls}")
    plt.xlabel("Fréquence")
    plt.ylabel("Mot")
    plt.tight_layout()
    plt.show()

## Prochaines étapes recommandées

1. Nettoyer les doublons sur `id` et vérifier les `title` quasi-identiques.
2. Créer des features supplémentaires (n-grams, TF-IDF, présence de ponctuation forte, etc.).
3. Split train/valid/test stratifié par `label` et éventuellement par `source`.
4. Entraîner un baseline (LogReg / LinearSVC) et comparer les performances intra-source vs cross-source.